# 🏥 Medical Classification & Recommendation System
## 1D-CNN for Respiratory Condition Diagnosis

This notebook builds a **1D Convolutional Neural Network** to classify patients into one of **5 respiratory conditions** based on vital signs and symptom features:

| Class | Condition |
|-------|-----------|
| 0 | Healthy |
| 1 | Cold |
| 2 | Flu |
| 3 | Bronchitis |
| 4 | Pneumonia |

**Input features** include heart rate, body temperature, blood pressure (systolic/diastolic), oxygen saturation (SpO₂), and binary-encoded symptoms (cough, fatigue, shortness of breath, chest pain, fever, sore throat, runny nose, headache).

---

## 1 · Environment Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_recall_fscore_support, multilabel_confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 2 · Synthetic Dataset Generation

We generate a clinically-inspired synthetic dataset with **8,000 samples**. Each condition has characteristic distributions for vital signs and symptom probabilities derived from medical literature ranges.

In [ ]:
def generate_medical_dataset(n_samples=8000, seed=42):
    """
    Generate a synthetic medical dataset with realistic vital-sign
    distributions and symptom probabilities per condition.

    Returns
    -------
    pd.DataFrame with columns:
        heart_rate, body_temperature, bp_systolic, bp_diastolic,
        spo2, cough, fatigue, shortness_of_breath, chest_pain,
        fever, sore_throat, runny_nose, headache, condition
    """
    rng = np.random.RandomState(seed)

    # Class names and intended proportions (slight imbalance)
    conditions = ['Healthy', 'Cold', 'Flu', 'Bronchitis', 'Pneumonia']
    proportions = np.array([0.30, 0.22, 0.20, 0.16, 0.12])  # imbalanced
    counts = (proportions * n_samples).astype(int)
    counts[-1] = n_samples - counts[:-1].sum()  # ensure exact total

    # Per-condition vital sign parameters: (mean, std)
    vital_params = {
        #                 HR          Temp(°F)     SysBP       DiaBP       SpO2
        'Healthy':    [(72, 6),    (98.6, 0.3), (120, 8),  (80, 5),   (98, 0.8)],
        'Cold':       [(78, 7),    (99.2, 0.5), (122, 9),  (82, 6),   (97, 1.0)],
        'Flu':        [(90, 10),   (101.5, 1.0),(125, 10), (84, 7),   (96, 1.2)],
        'Bronchitis': [(88, 9),    (100.5, 0.8),(128, 11), (85, 7),   (94, 1.5)],
        'Pneumonia':  [(100, 12),  (102.5, 1.2),(130, 12), (88, 8),   (91, 2.5)],
    }

    # Per-condition symptom probabilities
    symptom_names = [
        'cough', 'fatigue', 'shortness_of_breath', 'chest_pain',
        'fever', 'sore_throat', 'runny_nose', 'headache'
    ]
    symptom_probs = {
        'Healthy':    [0.05, 0.08, 0.02, 0.01, 0.02, 0.03, 0.05, 0.06],
        'Cold':       [0.60, 0.40, 0.08, 0.03, 0.15, 0.65, 0.75, 0.45],
        'Flu':        [0.55, 0.80, 0.20, 0.10, 0.85, 0.50, 0.35, 0.75],
        'Bronchitis': [0.85, 0.65, 0.55, 0.40, 0.50, 0.25, 0.15, 0.35],
        'Pneumonia':  [0.80, 0.85, 0.75, 0.60, 0.90, 0.20, 0.10, 0.50],
    }

    rows = []
    for cond, n in zip(conditions, counts):
        vp = vital_params[cond]
        sp = symptom_probs[cond]
        for _ in range(n):
            row = {
                'heart_rate':           np.clip(rng.normal(vp[0][0], vp[0][1]), 50, 150),
                'body_temperature':     np.clip(rng.normal(vp[1][0], vp[1][1]), 96.0, 106.0),
                'bp_systolic':          np.clip(rng.normal(vp[2][0], vp[2][1]), 90, 180),
                'bp_diastolic':         np.clip(rng.normal(vp[3][0], vp[3][1]), 60, 120),
                'spo2':                 np.clip(rng.normal(vp[4][0], vp[4][1]), 80, 100),
            }
            for sname, sprob in zip(symptom_names, sp):
                row[sname] = int(rng.random() < sprob)
            row['condition'] = cond
            rows.append(row)

    df = pd.DataFrame(rows)

    # Inject ~2 % missing values at random positions in vital signs
    vital_cols = ['heart_rate', 'body_temperature', 'bp_systolic', 'bp_diastolic', 'spo2']
    mask = rng.random(size=(len(df), len(vital_cols))) < 0.02
    for i, col in enumerate(vital_cols):
        df.loc[mask[:, i], col] = np.nan

    return df.sample(frac=1, random_state=seed).reset_index(drop=True)

df = generate_medical_dataset()
print(f"Dataset shape: {df.shape}")
df.head(10)

## 3 · Exploratory Data Analysis

In [ ]:
# --- 3.1  Class distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_counts = df['condition'].value_counts()
palette = sns.color_palette('Set2', n_colors=5)

axes[0].barh(class_counts.index, class_counts.values, color=palette)
axes[0].set_xlabel('Count')
axes[0].set_title('Class Distribution')
for i, v in enumerate(class_counts.values):
    axes[0].text(v + 20, i, str(v), va='center', fontweight='bold')

# --- 3.2  Missing value summary ---
missing = df.isnull().sum()
missing = missing[missing > 0]
axes[1].bar(missing.index, missing.values, color='salmon')
axes[1].set_ylabel('Missing Count')
axes[1].set_title('Missing Values per Feature')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# --- 3.3  Vital-sign distributions by condition ---
vital_cols = ['heart_rate', 'body_temperature', 'bp_systolic', 'bp_diastolic', 'spo2']
fig, axes = plt.subplots(1, 5, figsize=(24, 5))

for ax, col in zip(axes, vital_cols):
    for cond in df['condition'].unique():
        subset = df[df['condition'] == cond][col].dropna()
        ax.hist(subset, bins=30, alpha=0.5, label=cond, density=True)
    ax.set_title(col.replace('_', ' ').title())
    ax.legend(fontsize=7)

plt.suptitle('Vital Sign Distributions by Condition', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4 · Preprocessing Pipeline

In [ ]:
# --- 4.1  Handle missing values (median imputation per column) ---
for col in vital_cols:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)

assert df.isnull().sum().sum() == 0, "There should be no missing values after imputation."
print("✅ Missing values imputed (median strategy).")

In [ ]:
# --- 4.2  Encode target variable ---
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['condition'])

CLASS_NAMES = list(label_encoder.classes_)
NUM_CLASSES = len(CLASS_NAMES)

print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Label mapping: {dict(zip(CLASS_NAMES, label_encoder.transform(CLASS_NAMES)))}")

In [ ]:
# --- 4.3  Feature / target separation ---
symptom_cols = ['cough', 'fatigue', 'shortness_of_breath', 'chest_pain',
                'fever', 'sore_throat', 'runny_nose', 'headache']

feature_cols = vital_cols + symptom_cols
X = df[feature_cols].values.astype(np.float32)
y = df['label'].values

print(f"Feature matrix shape: {X.shape}")
print(f"Feature columns ({len(feature_cols)}): {feature_cols}")

In [ ]:
# --- 4.4  Stratified Train / Validation / Test split (70/15/15) ---
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.176,  # 0.176 × 0.85 ≈ 0.15 of total
    stratify=y_train_full, random_state=SEED
)

print(f"Train:  {X_train.shape[0]} samples")
print(f"Val:    {X_val.shape[0]} samples")
print(f"Test:   {X_test.shape[0]} samples")

In [ ]:
# --- 4.5  Feature scaling (fit on train only) ---
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print("✅ StandardScaler fitted on training set and applied to val/test.")

In [ ]:
# --- 4.6  One-Hot Encode targets for categorical cross-entropy ---
y_train_oh = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_val_oh   = keras.utils.to_categorical(y_val,   NUM_CLASSES)
y_test_oh  = keras.utils.to_categorical(y_test,  NUM_CLASSES)

print(f"One-hot target shape (train): {y_train_oh.shape}")

In [ ]:
# --- 4.7  Reshape for Conv1D  (samples, timesteps=features, channels=1) ---
NUM_FEATURES = X_train.shape[1]

X_train_cnn = X_train.reshape(-1, NUM_FEATURES, 1)
X_val_cnn   = X_val.reshape(-1, NUM_FEATURES, 1)
X_test_cnn  = X_test.reshape(-1, NUM_FEATURES, 1)

print(f"CNN input shape: {X_train_cnn.shape}  → (samples, {NUM_FEATURES} features, 1 channel)")

In [ ]:
# --- 4.8  Compute class weights to handle imbalance ---
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = {i: w for i, w in enumerate(class_weights_arr)}

print("Class weights:")
for cls_id, w in class_weight_dict.items():
    print(f"  {CLASS_NAMES[cls_id]:>12s} (class {cls_id}): {w:.4f}")

## 5 · 1D-CNN Model Architecture

In [ ]:
def build_1dcnn(input_shape, num_classes):
    """
    Build a 1D-CNN for tabular feature classification.

    Architecture
    ------------
    Conv1D (64, k=3) → BN → ReLU → Dropout(0.3)
    Conv1D (128, k=3) → BN → ReLU → Dropout(0.3)
    Conv1D (128, k=3) → BN → ReLU → GlobalMaxPool
    Dense(256) → BN → ReLU → Dropout(0.4)
    Dense(128) → BN → ReLU → Dropout(0.3)
    Dense(num_classes, softmax)
    """
    inp = layers.Input(shape=input_shape, name='input_features')

    # --- Conv Block 1 ---
    x = layers.Conv1D(64, kernel_size=3, padding='same',
                      kernel_regularizer=regularizers.l2(1e-4))(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)

    # --- Conv Block 2 ---
    x = layers.Conv1D(128, kernel_size=3, padding='same',
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)

    # --- Conv Block 3 ---
    x = layers.Conv1D(128, kernel_size=3, padding='same',
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.GlobalMaxPooling1D()(x)

    # --- Dense Head ---
    x = layers.Dense(256, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)

    out = layers.Dense(num_classes, activation='softmax', name='output')(x)

    model = keras.Model(inputs=inp, outputs=out, name='Medical_1DCNN')
    return model


model = build_1dcnn(
    input_shape=(NUM_FEATURES, 1),
    num_classes=NUM_CLASSES
)

model.summary()

In [ ]:
# Visualize architecture (optional – requires graphviz)
try:
    keras.utils.plot_model(
        model, to_file='model_architecture.png',
        show_shapes=True, show_layer_names=True, dpi=120
    )
    from IPython.display import Image
    display(Image('model_architecture.png'))
except Exception:
    print("(graphviz not available – skipping architecture diagram)")

## 6 · Compile & Train

In [ ]:
# --- 6.1  Compile ---
optimizer = keras.optimizers.Adam(learning_rate=1e-3)

model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print("✅ Model compiled.")

In [ ]:
# --- 6.2  Callbacks ---
cb_list = [
    callbacks.EarlyStopping(
        monitor='val_loss', patience=15,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=7, min_lr=1e-6, verbose=1
    ),
    callbacks.ModelCheckpoint(
        'best_model.keras', monitor='val_accuracy',
        save_best_only=True, verbose=0
    ),
]

In [ ]:
# --- 6.3  Train ---
EPOCHS = 100
BATCH_SIZE = 64

history = model.fit(
    X_train_cnn, y_train_oh,
    validation_data=(X_val_cnn, y_val_oh),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=cb_list,
    verbose=1
)

## 7 · Training History Visualization

In [ ]:
def plot_training_history(history):
    """Plot loss and accuracy curves for training vs. validation."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    epochs_range = range(1, len(history.history['loss']) + 1)

    # --- Loss ---
    axes[0].plot(epochs_range, history.history['loss'],
                 'o-', label='Train Loss', linewidth=2, markersize=3, color='#3498db')
    axes[0].plot(epochs_range, history.history['val_loss'],
                 's-', label='Val Loss', linewidth=2, markersize=3, color='#e74c3c')
    axes[0].set_title('Loss Curves', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Categorical Cross-Entropy')
    axes[0].legend(fontsize=12)
    axes[0].grid(True, alpha=0.3)

    # --- Accuracy ---
    axes[1].plot(epochs_range, history.history['accuracy'],
                 'o-', label='Train Accuracy', linewidth=2, markersize=3, color='#2ecc71')
    axes[1].plot(epochs_range, history.history['val_accuracy'],
                 's-', label='Val Accuracy', linewidth=2, markersize=3, color='#9b59b6')
    axes[1].set_title('Accuracy Curves', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend(fontsize=12)
    axes[1].grid(True, alpha=0.3)

    plt.suptitle('Training History', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("📊 Saved → training_history.png")


plot_training_history(history)

## 8 · Evaluation on Test Set

In [ ]:
# --- 8.1  Reload best model & evaluate ---
best_model = keras.models.load_model('best_model.keras')
test_loss, test_acc = best_model.evaluate(X_test_cnn, y_test_oh, verbose=0)
print(f"\n{'='*50}")
print(f"  Test Loss:     {test_loss:.4f}")
print(f"  Test Accuracy: {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"{'='*50}")

In [ ]:
# --- 8.2  Predictions ---
y_pred_proba = best_model.predict(X_test_cnn, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)

### 8.1 · Confusion Matrix

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names):
    """Plot a detailed confusion-matrix heatmap."""
    cm = confusion_matrix(y_true, y_pred)
    cm_pct = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    # Raw counts
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=class_names, yticklabels=class_names,
        ax=axes[0], cbar_kws={'label': 'Count'},
        linewidths=0.5, linecolor='white'
    )
    axes[0].set_title('Confusion Matrix (Counts)', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Predicted', fontsize=11)
    axes[0].set_ylabel('Actual', fontsize=11)

    # Percentages
    annot_pct = np.array([[f'{v:.1f}%' for v in row] for row in cm_pct])
    sns.heatmap(
        cm_pct, annot=annot_pct, fmt='', cmap='Oranges',
        xticklabels=class_names, yticklabels=class_names,
        ax=axes[1], cbar_kws={'label': 'Percentage'},
        linewidths=0.5, linecolor='white'
    )
    axes[1].set_title('Confusion Matrix (Row %)', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Predicted', fontsize=11)
    axes[1].set_ylabel('Actual', fontsize=11)

    plt.suptitle('5-Class Confusion Matrix — Test Set', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("📊 Saved → confusion_matrix.png")


plot_confusion_matrix(y_test, y_pred, CLASS_NAMES)

### 8.2 · Classification Report

In [ ]:
# --- Scikit-learn classification report ---
report = classification_report(
    y_test, y_pred,
    target_names=CLASS_NAMES, digits=4
)
print("\n" + "="*65)
print("       CLASSIFICATION REPORT  (Test Set)")
print("="*65)
print(report)

### 8.3 · Per-Class TP / TN / FP / FN Breakdown

In [ ]:
def per_class_metrics(y_true, y_pred, class_names):
    """
    Compute and display TP, TN, FP, FN, Precision, Recall, and F1-Score
    for every class using the One-vs-Rest confusion matrices.
    """
    mcm = multilabel_confusion_matrix(y_true, y_pred)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None
    )

    rows = []
    for i, cls in enumerate(class_names):
        tn, fp, fn, tp = mcm[i].ravel()
        rows.append({
            'Class': cls,
            'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
            'Precision': precision[i],
            'Recall': recall[i],
            'F1-Score': f1[i],
            'Support': support[i]
        })

    df_metrics = pd.DataFrame(rows)
    df_metrics = df_metrics.set_index('Class')

    # Overall (macro)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro'
    )

    print("\n" + "="*90)
    print("  PER-CLASS METRICS  (TP / TN / FP / FN + Precision / Recall / F1)")
    print("="*90)
    print(df_metrics.to_string())
    print("-"*90)
    print(f"  Macro Avg  →  Precision: {macro_p:.4f}  |  Recall: {macro_r:.4f}  |  F1: {macro_f1:.4f}")
    print("="*90)

    return df_metrics


df_metrics = per_class_metrics(y_test, y_pred, CLASS_NAMES)

In [ ]:
# --- Visual: Per-class Precision / Recall / F1 bar chart ---
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(NUM_CLASSES)
width = 0.25

bars1 = ax.bar(x - width, df_metrics['Precision'], width,
               label='Precision', color='#3498db', edgecolor='white')
bars2 = ax.bar(x,         df_metrics['Recall'],    width,
               label='Recall',    color='#2ecc71', edgecolor='white')
bars3 = ax.bar(x + width, df_metrics['F1-Score'],  width,
               label='F1-Score',  color='#e74c3c', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, fontsize=11)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Per-Class Precision, Recall & F1-Score', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.15)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 4), textcoords='offset points',
                    ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Saved → per_class_metrics.png")

## 9 · Medical Recommendation Engine

A rule-based recommendation module that maps predicted conditions to actionable clinical guidance.

In [ ]:
RECOMMENDATIONS = {
    'Healthy': {
        'severity': '🟢 None',
        'action': 'No immediate action required.',
        'advice': [
            'Continue regular health check-ups.',
            'Maintain a balanced diet and exercise routine.',
            'Stay hydrated and ensure adequate sleep (7-9 hrs).',
            'Keep vaccinations up to date.',
        ]
    },
    'Cold': {
        'severity': '🟡 Mild',
        'action': 'Self-care with OTC medications; consult a doctor if symptoms persist > 10 days.',
        'advice': [
            'Rest and drink plenty of fluids.',
            'Use saline nasal drops or a humidifier.',
            'OTC decongestants or antihistamines may provide relief.',
            'Avoid close contact with others to prevent spreading.',
        ]
    },
    'Flu': {
        'severity': '🟠 Moderate',
        'action': 'Seek medical evaluation; antiviral treatment within 48 hrs of symptom onset.',
        'advice': [
            'Bed rest is strongly recommended.',
            'Antiviral medications (e.g., oseltamivir) may shorten illness.',
            'Monitor for warning signs: difficulty breathing, chest pain.',
            'Annual flu vaccination is recommended for prevention.',
        ]
    },
    'Bronchitis': {
        'severity': '🟠 Moderate–High',
        'action': 'Medical consultation recommended; antibiotics if bacterial origin suspected.',
        'advice': [
            'Avoid smoking and irritants.',
            'Use a humidifier and stay hydrated.',
            'Cough suppressants or expectorants as directed.',
            'Seek urgent care if cough produces blood or lasts > 3 weeks.',
        ]
    },
    'Pneumonia': {
        'severity': '🔴 High',
        'action': 'Immediate medical attention required; possible hospitalization.',
        'advice': [
            'Antibiotics, antivirals, or antifungals depending on cause.',
            'Oxygen therapy may be needed if SpO₂ < 92%.',
            'Chest X-ray and blood tests recommended.',
            'Pneumococcal and flu vaccines reduce risk of pneumonia.',
        ]
    }
}


def get_recommendation(prediction_idx, probabilities=None):
    """
    Given a predicted class index (and optional probabilities),
    return a formatted medical recommendation.
    """
    cond = CLASS_NAMES[prediction_idx]
    rec = RECOMMENDATIONS[cond]
    confidence = probabilities[prediction_idx] * 100 if probabilities is not None else None

    print(f"\n{'━'*60}")
    print(f"  🏥  DIAGNOSIS:  {cond}")
    if confidence is not None:
        print(f"  📊  Confidence:  {confidence:.1f}%")
    print(f"  ⚠️   Severity:   {rec['severity']}")
    print(f"  🩺  Action:     {rec['action']}")
    print(f"\n  📋  Recommendations:")
    for i, advice in enumerate(rec['advice'], 1):
        print(f"      {i}. {advice}")
    print(f"{'━'*60}")

In [ ]:
# --- Demo: Show recommendations for first 5 test samples ---
print("\n" + "="*60)
print("   SAMPLE RECOMMENDATIONS  (first 5 test patients)")
print("="*60)

for i in range(5):
    actual = CLASS_NAMES[y_test[i]]
    pred_idx = y_pred[i]
    probs = y_pred_proba[i]

    print(f"\n🧑‍⚕️ Patient #{i+1}  |  Actual: {actual}")
    get_recommendation(pred_idx, probs)

## 10 · Interactive Prediction Function

In [ ]:
def predict_condition(heart_rate, body_temperature, bp_systolic, bp_diastolic,
                      spo2, cough, fatigue, shortness_of_breath, chest_pain,
                      fever, sore_throat, runny_nose, headache):
    """
    Predict a medical condition from raw patient features.

    Parameters
    ----------
    heart_rate : float         Body Temperature : float (°F)
    bp_systolic : float        bp_diastolic : float
    spo2 : float               cough, fatigue, ... : int (0 or 1)

    Returns
    -------
    Prints diagnosis and recommendation.
    """
    features = np.array([[
        heart_rate, body_temperature, bp_systolic, bp_diastolic, spo2,
        cough, fatigue, shortness_of_breath, chest_pain,
        fever, sore_throat, runny_nose, headache
    ]], dtype=np.float32)

    features_scaled = scaler.transform(features)
    features_cnn = features_scaled.reshape(1, NUM_FEATURES, 1)

    probs = best_model.predict(features_cnn, verbose=0)[0]
    pred_idx = np.argmax(probs)

    # Show probability distribution
    print("\n📊 Prediction Probabilities:")
    for j, cls in enumerate(CLASS_NAMES):
        bar = '█' * int(probs[j] * 40)
        print(f"  {cls:>12s}: {probs[j]:6.2%}  {bar}")

    get_recommendation(pred_idx, probs)


# --- Example usage ---
print("\n" + "="*60)
print("  INTERACTIVE PREDICTION DEMO")
print("="*60)

# Example: a patient with Flu-like symptoms
predict_condition(
    heart_rate=95, body_temperature=102.0,
    bp_systolic=128, bp_diastolic=85,
    spo2=95,
    cough=1, fatigue=1, shortness_of_breath=0, chest_pain=0,
    fever=1, sore_throat=1, runny_nose=0, headache=1
)

## 11 · Summary & Saved Artifacts

| Artifact | Description |
|----------|-------------|
| `best_model.keras` | Best model checkpoint (by val accuracy) |
| `training_history.png` | Loss & accuracy curves |
| `confusion_matrix.png` | 5-class confusion matrix heatmaps |
| `per_class_metrics.png` | Precision / Recall / F1 bar chart |
| `model_architecture.png` | Network architecture diagram (if graphviz available) |

---

### ⚠️ Disclaimer
This model is for **educational and research purposes only**. It should **not** be used for actual medical diagnosis. Always consult a qualified healthcare professional for medical decisions.